In [17]:
import os
import chromadb
from datasets import load_dataset
from google import genai

# Local environment configuration
CHROMA_PATH = r"chroma_db"

# 1. Connect to local ChromaDB instance
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="squad_benchmark")

# 2. Download and load the SQuAD dataset automatically from Hugging Face
print("Downloading SQuAD dataset from Hugging Face...")
raw_dataset = load_dataset("rajpurkar/squad", split="train")

# 3. Connect to the Gemini API Client using Google Cloud Vertex AI
# Point to your local service account credential key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gcp-key.json"

ai_client = genai.Client(
    vertexai=True,
    project="info-h512-project-rag-project",
    location="europe-west1"
)

documents = []
metadata = []
ids = []
embeddings = []

# To avoid hitting API quotas or saturating disk memory during testing,
# we index a clean sample slice of 500 unique context passages.
max_passages = 500
seen_contexts = set()

print(f"Processing and embedding the first {max_passages} unique SQuAD contexts...")

for i, entry in enumerate(raw_dataset):
    context_text = entry["context"]
    
    # SQuAD repeats identical context passages for multiple questions; filter for uniqueness
    if context_text in seen_contexts:
        continue
    seen_contexts.add(context_text)
    
    # Store text and unique ID mapping
    documents.append(context_text)
    doc_id = f"SQUAD_{len(seen_contexts)}"
    ids.append(doc_id)
    
    # Extract metadata provided natively by the dataset
    meta = {
        "title": entry["title"],
        "id": entry["id"]
    }
    metadata.append(meta)
    
    # Calculate vector embeddings using Vertex AI text embedding model
    response = ai_client.models.embed_content(
        model="text-embedding-004",
        contents=context_text
    )
    embeddings.append(response.embeddings[0].values)
    
    # Break early once we hit our sample target scale
    if len(seen_contexts) >= max_passages:
        break

# 4. Save the SQuAD text strings and vector coordinates into your local storage
print("Saving SQuAD records to local ChromaDB storage...")
collection.upsert(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadata,
    ids=ids
)
print(f"Successfully indexed {len(documents)} SQuAD passages!")

Processing and embedding the first 500 unique SQuAD contexts...


ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': "Permission 'aiplatform.endpoints.predict' denied on resource '//aiplatform.googleapis.com/projects/info-h512-project-rag-project/locations/europe-west1/publishers/google/models/text-embedding-004' (or it may not exist).", 'status': 'PERMISSION_DENIED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'IAM_PERMISSION_DENIED', 'domain': 'aiplatform.googleapis.com', 'metadata': {'permission': 'aiplatform.endpoints.predict', 'resource': 'projects/info-h512-project-rag-project/locations/europe-west1/publishers/google/models/text-embedding-004'}}]}}

In [ ]:
import os
import numpy as np
import chromadb
from google import genai
from google.genai import types
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Local environment configuration
CHROMA_PATH = r"chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="growing_vegetables")

# Initialize official Gemini API client
ai_client = genai.Client()

# Fetch all records into memory to build the local BM25 engine
all_db_data = collection.get(include=['documents', 'embeddings'])
all_documents = all_db_data['documents']
all_embeddings = all_db_data['embeddings']

# Tokenize corpus for BM25 keyword matching
tokenized_corpus = [doc.lower().split(" ") for doc in all_documents]
bm25 = BM25Okapi(tokenized_corpus)


# ==========================================
# RAG RETRIEVAL STRATEGIES
# ==========================================

def execute_dense_search(query_text, n_results=10):
    """Executes purely semantic (vector-based) search."""
    # Compute the query vector embedding
    query_emb = ai_client.models.embed_content(
        model="text-embedding-004",
        contents=query_text
    ).embeddings[0].values
    
    # Query local vector database
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        include=['documents']
    )
    return results['documents'][0] if results['documents'] else []


def execute_hybrid_search(query_text, top_k=5):
    """METHOD 1: Hybrid Search combining BM25 and Dense Vectors using Reciprocal Rank Fusion (RRF)."""
    # A. Retrieve top 10 semantic candidates
    dense_results = execute_dense_search(query_text, n_results=10)
    
    # B. Retrieve top 10 keyword candidates using BM25
    tokenized_query = query_text.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:10]
    sparse_results = [all_documents[idx] for idx in top_bm25_indices]
    
    # C. Blend lists using Reciprocal Rank Fusion (RRF) algorithm
    rrf_scores = {}
    def add_ranks(doc_list):
        for rank, doc in enumerate(doc_list):
            if doc not in rrf_scores:
                rrf_scores[doc] = 0.0
            # Standard RRF formula using constant = 60
            rrf_scores[doc] += 1.0 / (60 + (rank + 1))
            
    add_ranks(dense_results)
    add_ranks(sparse_results)
    
    # Sort candidates based on combined RRF score
    sorted_docs = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [doc for doc, score in sorted_docs[:top_k]]


def execute_reranking(query_text, top_k=3):
    """METHOD 2: Two-stage retrieval using an open-source Cross-Encoder re-ranker."""
    # 1. Fetch an expanded candidate pool (e.g., 15 documents) via Hybrid Search
    candidate_docs = execute_hybrid_search(query_text, top_k=15)
    
    if not candidate_docs:
        return []
        
    # 2. Initialize a lightweight local Cross-Encoder re-ranking filter
    # Runs locally on your CPU/GPU in milliseconds
    reranker = CrossEncoder("BAAI/bge-reranker-base")
    
    # 3. Compute precise, non-pooled relevance scores between the query and each chunk
    pairs = [[query_text, doc] for doc in candidate_docs]
    scores = reranker.predict(pairs)
    
    # 4. Re-sort candidates and slice the top-K highest signal results
    sorted_indices = np.argsort(scores)[::-1]
    return [candidate_docs[idx] for idx in sorted_indices[:top_k]]


def execute_hyde_search(query_text, top_k=5):
    """METHOD 3: Hypothetical Document Embedding (HyDE)."""
    # 1. Leverage the LLM to generate a speculative, ideal answer paragraph
    hyde_prompt = f"Write a brief, hypothetical paragraph answering the following question about vegetable gardening: '{query_text}'"
    hyde_response = ai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=hyde_prompt
    )
    hypothetical_answer = hyde_response.text
    print(f"\n[HyDE] Generated Hypothetical Answer:\n{hypothetical_answer}\n")
    
    # 2. Use the generated hypothetical answer to query the hybrid vector store
    return execute_hybrid_search(hypothetical_answer, top_k=top_k)


# ==========================================
# APPLICATION RUNTIME PIPELINE
# ==========================================

if __name__ == '__main__':
    user_query = input("What do you want to know about growing vegetables?\n\n")
    
    print("\nSelect the RAG routing architecture to evaluate:")
    print("1: Baseline Hybrid Search (BM25 + Dense Vector)")
    print("2: Two-Stage Hybrid Search + Re-ranking Filter (Cross-Encoder)")
    print("3: HyDE (Hypothetical Document Embedding) + Hybrid Search")
    
    choice = input("\nEnter your selection (1, 2, or 3): ")
    
    # Routing processing logic based on user choice
    if choice == "1":
        print("\nExecuting baseline Hybrid Search pipeline...")
        retrieved_contexts = execute_hybrid_search(user_query, top_k=3)
    elif choice == "2":
        print("\nExecuting Cross-Encoder Re-ranking pipeline...")
        retrieved_contexts = execute_reranking(user_query, top_k=3)
    elif choice == "3":
        print("\nExecuting HyDE pipeline expansion...")
        retrieved_contexts = execute_hyde_search(user_query, top_k=3)
    else:
        print("Invalid option selected. Defaulting to baseline Hybrid Search.")
        retrieved_contexts = execute_hybrid_search(user_query, top_k=3)
        
    # Standardize retrieved text chunks into a unified context string
    context_str = "\n---\n".join(retrieved_contexts)
    
    # Construct the strict system prompt injection pattern
    system_prompt = f"""
You are a precise academic assistant. You answer questions based strictly on the knowledge 
I am providing you. You do not use your internal knowledge, and you do not make things up.
If the answer cannot be found in the data, simply say: I don't know
--------------------
The data:
{context_str}
"""

    # Dispatch request to the text generation engine (Gemini 2.5 Flash)
    response = ai_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(role="user", parts=[types.Part.from_text(text=system_prompt)]),
            types.Content(role="user", parts=[types.Part.from_text(text=user_query)])
        ]
    )
    
    print("\n\n--------------------- EXTRACTED CONTEXT ---------------------\n")
    print(context_str if context_str else "No relevant context found.")
    print("\n--------------------- GEMINI GENERATION ---------------------\n")
    print(response.text)